# 20A Appendix: Walk Counting, Random-Graph Degrees and the Pareto Views


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/20A-Appendix-Graph-Derivations.ipynb)

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| code-fold: true
#| code-summary: "Imports and the lecture's five-node example"
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import pareto

from slideUtilities import OKABE_ITO


def draw_small(G, pos, color, title, ax=None, **kwargs):
    """Draw one of the lecture's small labelled example graphs."""
    if ax is None:
        plt.figure(figsize=(6, 4))
        ax = plt.gca()
    nx.draw_networkx(G, pos=pos, ax=ax, node_size=300, edge_color='k',
                     node_color=color, with_labels=True,
                     alpha=1, linewidths=2, **kwargs)
    ax.set_title(title, size=16)
    ax.axis('off')


edges = [(1, 2), (1, 3), (2, 3), (3, 4), (3, 5)]
G = nx.Graph()
G.add_edges_from(edges)
pos_u = nx.spring_layout(G, seed=1)
A = nx.to_numpy_array(G, nodelist=[1, 2, 3, 4, 5], dtype=int)

This appendix holds the derivations behind [Lecture 20](20-Networks-I.qmd): why the entries of $A^2$ count walks, the degree distribution of the $G(n, p)$ random graph, the pdf and the three views of the Pareto distribution, how an empirical CCDF is built, the node-by-node clustering table for the five-node example, the path-length check for the ring lattice, and the spectral layout of the football network. The lecture *states* each result and applies it to a tiny graph; this is where it is worked out. The quiz asks you to read an adjacency matrix, compute one entry of $A^2$ and one clustering coefficient on a small drawn graph, and tell a heavy-tailed CCDF from a $G(n, p)$-like one; it does not ask for these derivations.

The spectral layout at the end previews the optional module [Graphs II: Centrality and Clustering](21-Networks-II-Centrality-Clustering.qmd), which is not on the quiz.

## Why $A^2$ Counts Walks of Length 2

Let $A$ be the adjacency matrix of a graph on nodes $1, \dots, n$. By the definition of matrix multiplication,

$$
(A^2)_{ij} = \sum_{k=1}^{n} A_{ik} A_{kj}.
$$

Every factor is 0 or 1, so each term of the sum is 1 exactly when $A_{ik} = 1$ **and** $A_{kj} = 1$: when $(i, k)$ and $(k, j)$ are both edges, which is to say when $i \to k \to j$ is a **walk of length 2**. Summing over $k$ counts the walks of length 2 from $i$ to $j$, one per intermediate node $k$. It is a count of *walks*, not paths: when $i = j$ the walk $i \to k \to i$ goes out along an edge and straight back, revisiting $i$, which a path may not do.

In [ ]:
#| code-fold: false
print(A @ A)

* $(A^2)_{12} = 1$: the single walk $1 \to 3 \to 2$. Node 3 is the only common neighbor of nodes 1 and 2, so there is exactly one intermediate $k$ that works.
* $(A^2)_{45} = 1$: the walk $4 \to 3 \to 5$. Nodes 4 and 5 are not adjacent, but they are two steps apart, and $A^2$ sees that.
* The diagonal is the degree: $(A^2)_{ii} = \sum_k A_{ik} A_{ki} = \sum_k A_{ik}^2 = \sum_k A_{ik}$, since every $A_{ik}$ is 0 or 1 and $A$ is symmetric. So $(A^2)_{33} = 4$ counts the walks $3 \to k \to 3$, one per edge at node 3.

The same argument repeats: $(A^3)_{ij} = \sum_k (A^2)_{ik} A_{kj}$ counts walks of length 3, and in general $(A^m)_{ij}$ is the number of walks of length $m$ from $i$ to $j$. For a directed graph the same holds with the arrows respected, since $A_{ik} = 1$ only for an edge $i \to k$. Powers of $A$ counting walks is where the centrality measures of the optional Graphs II module come from.

## The Degree Distribution of $G(n, p)$

In $G(n, p)$ every node is potentially connected to each of the $n - 1$ other nodes, independently with probability $p$. The expected degree, call it $\langle k \rangle$, is therefore

$$
\langle k \rangle = p \cdot (n - 1),
$$

which for large $n$ is approximately

$$
\langle k \rangle \approx np.
$$

More than the mean is known. The degree of a node is a sum of $n - 1$ independent coin flips, each coming up heads with probability $p$, so it is $\text{Binomial}(n - 1, p)$. When $n$ is large and $p$ is small with $np$ moderate, which is the sparse regime real networks live in, the binomial is approximately $\text{Poisson}(np)$. Both distributions are **concentrated** around their mean: the probability that a node has degree many times $np$ falls off exponentially, so a node with five or ten times the average degree is vanishingly rare. $G(n, p)$ has **no hubs**. That is the property real networks most often violate, and the reason the lecture plots degree distributions on log–log axes, where the difference is impossible to miss.

## Three Views of the Pareto Distribution

The simplest power law is the Pareto distribution. If $X$ is Pareto with scale $k$ and shape $\alpha$, then

$$
P(X > x) =
\begin{cases}
\left(\frac{k}{x}\right)^{\alpha} & x \geq k, \\
1 & x < k,
\end{cases}
$$

where $k$ is the minimum value of $X$ (the **scale parameter**, the smallest number of neighbors a node can have) and $\alpha$ is the exponent (the **shape parameter**). The pdf is

$$
p(x) =
\begin{cases}
\frac{\alpha k^{\alpha}}{x^{\alpha + 1}} &  x \geq k, \\
0 &  x < k.
\end{cases}
$$

The Pareto distribution was introduced in the [Probability and Statistics Refresher](A6-Probability-Statistics-Refresher.qmd). Here it is drawn three ways, with $k = 1$ and $\alpha = 1, 2, 3$.

In [ ]:
#| fig-align: center
alphas = [1, 2, 3]
colors = OKABE_ITO[:3]
x_lin = np.linspace(0.5, 10, 500)
x_log = np.logspace(0, 2, 200)   # 1 to 100: two decades

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for alpha, c in zip(alphas, colors):
    axes[0].plot(x_lin, pareto.sf(x_lin, alpha), color=c, lw=2, label=f'$\\alpha={alpha}$')
    axes[1].plot(x_lin, pareto.pdf(x_lin, alpha), color=c, lw=2, label=f'$\\alpha={alpha}$')
    axes[2].loglog(x_log, pareto.pdf(x_log, alpha), color=c, lw=2, label=f'$\\alpha={alpha}$')

axes[0].set(title='CCDF $P(X > x)$, linear axes', xlabel='$x$', ylabel='$P(X > x)$')
axes[1].set(title='pdf $p(x)$, linear axes', xlabel='$x$', ylabel='$p(x)$')
axes[2].set(title='pdf $p(x)$, log–log axes', xlabel='$x$', ylabel='$p(x)$')
for ax in axes:
    ax.legend()
plt.suptitle('Pareto distribution with $k = 1$', size=15)
plt.tight_layout()
plt.show()

On linear axes (left and middle) all you can see is a sharp drop and a long, flat tail hugging the axis. On log–log axes (right) each curve is a **straight line**, and the steeper lines are the larger $\alpha$:

$$
\log p(x) = \log(\alpha k^{\alpha}) - (\alpha + 1) \log x,
$$

a line in $\log x$ with slope $-(\alpha + 1)$. The CCDF is a straight line too, $\log P(X > x) = \alpha \log k - \alpha \log x$, with slope $-\alpha$. So the test for a power law is: plot the CCDF on log–log axes and see whether it is straight.

In a distribution like this almost all values are very small, but a non-negligible fraction are *very* large. [Vilfredo Pareto](https://en.wikipedia.org/wiki/Vilfredo_Pareto) originally used it to describe the allocation of wealth among individuals; the sizes of sand particles are also approximately Pareto-distributed. For node degree it means that most nodes have **few neighbors**, but an important small subset of nodes have **many, many neighbors**: hubs, which $G(n, p)$ never produces.

## The Empirical CCDF

To see a heavy tail in a real graph you do not fit anything. Sort the degrees, and for each distinct degree $x$ plot the fraction of nodes whose degree is **at least** $x$, on log–log axes. That is what the lecture's `empirical_ccdf` function does:

In [ ]:
#| code-fold: false
def empirical_ccdf(graph):
    # Fraction of nodes with degree >= each sorted degree. Using >= rather
    # than > keeps the largest degree off zero, which a log axis cannot show.
    d = np.sort([k for _, k in graph.degree()])
    frac = 1 - np.arange(len(d)) / len(d)
    keep = d > 0
    return d[keep], frac[keep]

d, frac = empirical_ccdf(G)
for degree, fraction in zip(d, frac):
    print(f"degree {degree}: fraction of nodes with degree >= {degree} is {fraction:.2f}")

Two details. The plotted quantity is $P(X \ge x)$ rather than the $P(X > x)$ of the definition: with $>$ the largest degree would have fraction 0, and a log axis cannot show 0, so the single most important point, the biggest hub, would vanish from the plot. The two differ only by one node's worth at each step and the shape of the tail is the same. And nodes of degree 0 are dropped for the same reason, since $\log 0$ is undefined on the $x$-axis.

The CCDF is preferred to a binned histogram because the tail is a handful of nodes. A histogram chops them into bins that hold one node or none, which on log–log axes looks like noise; the CCDF accumulates, so every node contributes and the tail stays smooth. On the lecture's 1,000-node Barabási–Albert graph the CCDF runs in close to a straight line out to a single node of degree 100, while the same-density $G(n, p)$ falls off a cliff before degree 15, exactly the concentration the previous section predicts.

## The Clustering Coefficient of the Example Graph

The clustering coefficient measures the probability that two of your neighbors are connected. A **triangle** is a set of three mutually connected nodes. The first definition is a ratio of totals,

$$
C^{(1)} = \frac{\sum_i \text{number of triangles that include node } i}{\sum_i \text{number of pairs of neighbors of node }i},
$$

the probability that a **random pair** of neighbors, anywhere in the graph, is connected. Node by node on the five-node example:

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(4, 3))
draw_small(G, pos_u, 'lightcoral', 'Triangle 1–2–3, plus 3–4, 3–5', ax=ax)
ax.title.set_size(12)
plt.show()

| Node | Neighbors | Pairs $\binom{d}{2}$ | Triangles | Ratio |
|:---:|:---:|:---:|:---:|:---:|
| 1 | {2, 3} | 1 | 1 | 1 |
| 2 | {1, 3} | 1 | 1 | 1 |
| 3 | {1, 2, 4, 5} | 6 | 1 | 1/6 |
| 4 | {3} | 0 | 0 | 0 |
| 5 | {3} | 0 | 0 | 0 |
| **Total** | | **8** | **3** | |


$C^{(1)} = 3/8 = 0.375$. Node 3 has $\binom{4}{2} = 6$ pairs of neighbors, but only $\{1, 2\}$ is connected. Nodes 4 and 5 each have a single neighbor, so they contribute no pairs and no triangles.

The second definition is the mean of the per-node ratios,

$$
C^{(2)} = \frac{1}{n} \sum_i \frac{\text{number of triangles that include node } i}{\text{number of pairs of neighbors of node }i},
$$

the probability that the neighbors of a **random node** are connected. A node with degree less than 2 has no pairs of neighbors; by convention its ratio is 0. Here nodes 4 and 5 contribute 0, so

$$
C^{(2)} = \frac{1}{5} \left(1 + 1 + \frac{1}{6} + 0 + 0\right) = \frac{13}{30} = 0.433.
$$

In [ ]:
#| code-fold: false
print(f"transitivity       (C1) = {nx.transitivity(G):.3f}")
print(f"average_clustering (C2) = {nx.average_clustering(G):.3f}")
print("per-node clustering:", {k: round(v, 3) for k, v in nx.clustering(G).items()})

In NetworkX, $C^{(1)}$ is `nx.transitivity` and $C^{(2)}$ is `nx.average_clustering`. They disagree because $C^{(1)}$ counts pairs: node 3 holds six of the eight pairs, so its poor one-in-six ratio dominates. $C^{(2)}$ gives every node one vote, so nodes 1 and 2, with perfect scores, pull it up. $C^{(1)}$ gives more weight to high-degree nodes, which have many pairs of neighbors; a network with a few giant hubs whose neighbors are mostly strangers to each other has a much smaller $C^{(1)}$ than $C^{(2)}$.

In a $G(n, p)$ random graph the probability that two of your neighbors are connected is the same as the probability that *any* two nodes are connected, so $C^{(1)} = C^{(2)} = p$.

## Path Length in the Ring Lattice

In the ring lattice each node is joined to its 4 nearest neighbors around a ring, so it has $\binom{4}{2} = 6$ pairs of neighbors, of which 3 are connected: $C = 0.5$. The average shortest path length is about $n/8$, because on average your target is a quarter of the way around the ring and you travel in hops of 2:

In [ ]:
#| code-fold: false
for n_ring in [20, 100, 1000]:
    ring = nx.watts_strogatz_graph(n_ring, 4, 0)
    print(f"n = {n_ring:4d}: C = {nx.average_clustering(ring):.2f}, "
          f"L = {nx.average_shortest_path_length(ring):6.1f}, "
          f"n/8 = {n_ring / 8:6.1f}")

The path length grows linearly with $n$: if the number of nodes is large, the average path length is large. That is the tension the Watts–Strogatz rewiring experiment in the lecture resolves, by showing that a few long-range edges collapse $L$ while barely touching $C$.

## Spectral Layout

The lecture draws the football network with a spring layout. A third option, the **spectral layout**, places each node at coordinates taken from eigenvectors of a matrix built from $A$ (the graph Laplacian). The optional module [Graphs II: Centrality and Clustering](21-Networks-II-Centrality-Clustering.qmd) defines it and explains why those eigenvectors separate the clusters; it is not on the quiz. Here is what it looks like.

In [ ]:
#| code-fold: true
#| code-summary: "Load the football network and its same-density random graph (as in the lecture)"
import os, urllib.request

def data_file(path):
    """Course data file: use a local copy (here, or in the repo root above a
    notebook in jupyter_notebooks/), else fetch it from the Materials repo (Colab)."""
    for prefix in ('', '../'):
        if os.path.exists(prefix + path):
            return prefix + path
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/' + path, path)
    return path

football = nx.read_gml(data_file('data/football.gml'))
n = football.number_of_nodes()
p = nx.density(football)          # e / (n choose 2)
F_random = nx.erdos_renyi_graph(n, p, seed=0)

conferences = ['Atlantic Coast', 'Big East', 'Big Ten', 'Big Twelve',
               'Conference USA', 'Independents', 'Mid-American', 'Mountain West',
               'Pacific Ten', 'Southeastern', 'Sun Belt', 'Western Athletic']
# 11 conferences need 11 colors: Okabe-Ito plus three dark tones.
# Independents (value 5) are not a conference, so they are grey.
conf_palette = OKABE_ITO + ['#882255', '#999933', '#8C564B']
conf_colors = [conf_palette[i] for i in range(5)] + ['#BBBBBB'] + \
              [conf_palette[i] for i in range(5, 11)]

conf = [football.nodes[team]['value'] for team in football.nodes()]
node_colors = [conf_colors[c] for c in conf]

def draw_pair(layout, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, graph, name in [(ax1, football, 'Title 1 Football'),
                            (ax2, F_random, 'Same Density Random')]:
        nx.draw_networkx(graph, ax=ax, pos=layout(graph), node_size=35,
                         node_color=node_colors, edge_color='lightgray',
                         with_labels=False, linewidths=0.5)
        ax.set_title(f'{name} -- {title}', size=15)
        ax.axis('off')
    return fig

In [ ]:
#| fig-align: center
draw_pair(nx.spectral_layout, 'Spectral Layout')
plt.show()

On the left the conferences pull apart into separate arms; on the right the same-density random graph collapses into a dense knot with a few stragglers. With a legend:

In [ ]:
#| fig-align: center
fig, ax = plt.subplots(figsize=(9, 6.5))
nx.draw_networkx(football, ax=ax, pos=nx.spectral_layout(football), node_size=40,
                 node_color=node_colors, edge_color='lightgray',
                 with_labels=False, linewidths=0.5)
handles = [plt.Line2D([], [], marker='o', ls='', color=conf_colors[i], label=name)
           for i, name in enumerate(conferences)]
ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=10)
ax.set_title('Title 1 Football -- Spectral Layout', size=15)
ax.axis('off')
plt.tight_layout()
plt.show()

The clusters of teams are the conferences. A few overlap (Conference USA and Sun Belt in the lower middle), and the grey independents are scattered, which is what you would expect for teams without a conference schedule. There are not many high-degree nodes: no hubs, consistent with the lecture's degree histogram.